# 07 — Sales Decomposition & Attribution

> **Objective**: Combine outputs from all four MMM frameworks into a unified sales decomposition and attribution view.

**Key Topics**:
- Waterfall chart: Baseline → each channel → Total
- Stacked area decomposition over time
- Baseline vs. incremental (media-driven) split
- Macro-adjusted baseline decomposition (festival, CPI, distribution)
- Framework consensus: where do models agree/disagree?

**Data**: Model outputs from notebooks 02, 04, 05, 06

## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import json
import os
import warnings

warnings.filterwarnings('ignore')

# Create output directory
os.makedirs('../outputs/figures', exist_ok=True)

# Load national weekly data
data = pd.read_csv('../data/processed/mmm_national_weekly.csv')
data['date'] = pd.to_datetime(data['date'])
print(f"Data shape: {data.shape}")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")
print(f"Total sales: ₹{data['sales'].sum()/1e6:.2f}M")

Data shape: (156, 13)
Date range: 2022-07-04 00:00:00 to 2025-06-23 00:00:00
Total sales: ₹371.77M


## 1. Load Decomposition Results from All Frameworks

In [2]:
# =============================================================================
# Load decomposition data from all frameworks
# =============================================================================

# Channel list
channels = ['tv', 'youtube', 'facebook', 'instagram', 'print_media', 'radio']
channel_display = {'tv': 'TV', 'youtube': 'YouTube', 'facebook': 'Facebook', 
                   'instagram': 'Instagram', 'print_media': 'Print', 'radio': 'Radio'}

# Total sales for the period
total_sales = data['sales'].sum()
print(f"Total sales (156 weeks): ₹{total_sales/1e6:.2f}M")

# -----------------------------------------------------------------------------
# 1. Classical MMM (Ridge) - from classical_roas.parquet
# -----------------------------------------------------------------------------
classical_roas = pd.read_parquet('../outputs/models/classical_roas.parquet')
classical_roas = classical_roas.set_index('Channel')

# Calculate contributions (ROAS * spend)
classical_decomp = {}
for ch in channels:
    if ch in classical_roas.index:
        roas = classical_roas.loc[ch, 'ROAS']
        spend = data[ch].sum()
        contrib = roas * spend
        classical_decomp[ch] = contrib
    else:
        classical_decomp[ch] = 0

classical_total_media = sum(classical_decomp.values())
classical_baseline = total_sales - classical_total_media

print(f"\nClassical MMM:")
print(f"  Baseline: ₹{classical_baseline/1e6:.2f}M ({classical_baseline/total_sales*100:.1f}%)")
print(f"  Media-driven: ₹{classical_total_media/1e6:.2f}M ({classical_total_media/total_sales*100:.1f}%)")

# -----------------------------------------------------------------------------
# 2. Bayesian (PyMC) - from bayesian_roas.parquet
# -----------------------------------------------------------------------------
bayesian_roas = pd.read_parquet('../outputs/models/bayesian_roas.parquet')
bayesian_roas = bayesian_roas.set_index('Channel')

bayesian_decomp = {}
for ch in channels:
    ch_name = ch if ch != 'print_media' else 'Print'
    if ch_name in bayesian_roas.index:
        roas = bayesian_roas.loc[ch_name, 'ROAS_mean']
        spend = data[ch].sum()
        contrib = roas * spend
        bayesian_decomp[ch] = contrib
    else:
        bayesian_decomp[ch] = 0

bayesian_total_media = sum(bayesian_decomp.values())
bayesian_baseline = total_sales - bayesian_total_media

print(f"\nBayesian MMM (PyMC):")
print(f"  Baseline: ₹{bayesian_baseline/1e6:.2f}M ({bayesian_baseline/total_sales*100:.1f}%)")
print(f"  Media-driven: ₹{bayesian_total_media/1e6:.2f}M ({bayesian_total_media/total_sales*100:.1f}%)")

# -----------------------------------------------------------------------------
# 3. Robyn (R) - from robyn_roas.csv
# -----------------------------------------------------------------------------
robyn_roas = pd.read_csv('../outputs/models/robyn_roas.csv')
robyn_roas = robyn_roas.set_index('rn')

robyn_decomp = {}
for ch in channels:
    if ch in robyn_roas.index:
        contrib = robyn_roas.loc[ch, 'xDecompAgg']  # Already calculated contribution
        robyn_decomp[ch] = contrib
    else:
        robyn_decomp[ch] = 0

robyn_total_media = sum(robyn_decomp.values())
robyn_baseline = total_sales - robyn_total_media

print(f"\nRobyn MMM:")
print(f"  Baseline: ₹{robyn_baseline/1e6:.2f}M ({robyn_baseline/total_sales*100:.1f}%)")
print(f"  Media-driven: ₹{robyn_total_media/1e6:.2f}M ({robyn_total_media/total_sales*100:.1f}%)")

# -----------------------------------------------------------------------------
# 4. Meridian (Google) - from meridian_roas.parquet
# -----------------------------------------------------------------------------
meridian_roas = pd.read_parquet('../outputs/models/meridian_roas.parquet')
meridian_roas = meridian_roas.set_index('Channel')

meridian_decomp = {}
for ch in channels:
    ch_name = ch if ch != 'print_media' else 'Print'
    if ch_name in meridian_roas.index:
        roas = meridian_roas.loc[ch_name, 'ROAS_mean']
        spend = data[ch].sum()
        contrib = roas * spend
        meridian_decomp[ch] = contrib
    else:
        meridian_decomp[ch] = 0

meridian_total_media = sum(meridian_decomp.values())
meridian_baseline = total_sales - meridian_total_media

print(f"\nMeridian MMM:")
print(f"  Baseline: ₹{meridian_baseline/1e6:.2f}M ({meridian_baseline/total_sales*100:.1f}%)")
print(f"  Media-driven: ₹{meridian_total_media/1e6:.2f}M ({meridian_total_media/total_sales*100:.1f}%)")

Total sales (156 weeks): ₹371.77M

Classical MMM:
  Baseline: ₹371.77M (100.0%)
  Media-driven: ₹0.00M (0.0%)

Bayesian MMM (PyMC):
  Baseline: ₹297.75M (80.1%)
  Media-driven: ₹74.02M (19.9%)

Robyn MMM:
  Baseline: ₹366.47M (98.6%)
  Media-driven: ₹5.30M (1.4%)

Meridian MMM:
  Baseline: ₹370.39M (99.6%)
  Media-driven: ₹1.38M (0.4%)


## 2. Baseline vs. Media-Driven Split

In [3]:
# =============================================================================
# Baseline vs. Media-Driven Split - Comparison Table
# =============================================================================

# Build comparison DataFrame
comparison_data = {
    'Framework': ['Classical (Ridge)', 'Bayesian (PyMC)', 'Robyn (R)', 'Meridian (Google)'],
    'Baseline (₹M)': [classical_baseline/1e6, bayesian_baseline/1e6, robyn_baseline/1e6, meridian_baseline/1e6],
    'Baseline (%)': [classical_baseline/total_sales*100, bayesian_baseline/total_sales*100, 
                     robyn_baseline/total_sales*100, meridian_baseline/total_sales*100],
    'Media-Driven (₹M)': [classical_total_media/1e6, bayesian_total_media/1e6, 
                          robyn_total_media/1e6, meridian_total_media/1e6],
    'Media-Driven (%)': [classical_total_media/total_sales*100, bayesian_total_media/total_sales*100,
                         robyn_total_media/total_sales*100, meridian_total_media/total_sales*100]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(2)
print("=" * 80)
print("BASELINE VS MEDIA-DRIVEN SPLIT ACROSS FRAMEWORKS")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Average across frameworks
avg_baseline = (classical_baseline + bayesian_baseline + robyn_baseline + meridian_baseline) / 4 / total_sales * 100
avg_media = (classical_total_media + bayesian_total_media + robyn_total_media + meridian_total_media) / 4 / total_sales * 100

print(f"\n📊 Average across frameworks:")
print(f"   Baseline: {avg_baseline:.1f}%")
print(f"   Media-Driven: {avg_media:.1f}%")

BASELINE VS MEDIA-DRIVEN SPLIT ACROSS FRAMEWORKS
        Framework  Baseline (₹M)  Baseline (%)  Media-Driven (₹M)  Media-Driven (%)
Classical (Ridge)         371.77        100.00               0.00              0.00
  Bayesian (PyMC)         297.75         80.09              74.02             19.91
        Robyn (R)         366.47         98.58               5.30              1.42
Meridian (Google)         370.39         99.63               1.38              0.37

📊 Average across frameworks:
   Baseline: 94.6%
   Media-Driven: 5.4%


## 3. Waterfall Chart — Classical MMM

In [4]:
# =============================================================================
# Waterfall Chart - Classical MMM Decomposition
# =============================================================================

# Build waterfall data for Classical MMM
waterfall_components = ['Baseline'] + [channel_display.get(ch, ch) for ch in channels]
waterfall_values = [classical_baseline] + [classical_decomp.get(ch, 0) for ch in channels]

# Calculate running total for waterfall
running_total = []
cumsum = 0
for i, val in enumerate(waterfall_values):
    if i == 0:
        running_total.append(0)  # Starting point
        cumsum = val
    else:
        running_total.append(cumsum)
        cumsum += val

# Add final total
waterfall_components.append('Total')
waterfall_values.append(sum(waterfall_values))
running_total.append(0)

# Create waterfall chart
fig = go.Figure(go.Waterfall(
    name = "Sales Decomposition",
    orientation = "v",
    measure = ["relative"] * (len(waterfall_components) - 1) + ["total"],
    x = waterfall_components,
    textposition = "outside",
    text = [f"₹{v/1e6:.1f}M" if v > 0 else f"₹{v/1e6:.1f}M" for v in waterfall_values],
    y = waterfall_values,
    connector = {"line": {"color": "rgb(63, 63, 63)"}},
    decreasing = {"marker": {"color": "#E15759"}},
    increasing = {"marker": {"color": "#4E79A7"}},
    totals = {"marker": {"color": "#59A14F"}}
))

fig.update_layout(
    title = "Sales Decomposition - Classical MMM (Ridge)",
    showlegend = False,
    height = 500,
    yaxis_title = "Sales (₹ Million)",
    template = "plotly_white"
)

fig.write_image("../outputs/figures/decomposition_waterfall.png", scale=2)
fig.show()
print("✅ Saved: ../outputs/figures/decomposition_waterfall.png")

✅ Saved: ../outputs/figures/decomposition_waterfall.png


## 4. Stacked Area Chart Over Time

In [5]:
# =============================================================================
# Stacked Area Chart - Weekly Decomposition Over Time
# =============================================================================

# Calculate weekly contributions using Classical MMM coefficients
# For simplicity, we'll use proportional allocation based on ROAS

# Get spend and calculate contribution ratios
weekly_decomp = pd.DataFrame({'date': data['date'], 'sales': data['sales']})

# Calculate baseline (intercept + control variables effect)
# Using simplified approach: baseline = sales - media contribution
# For each week, allocate based on channel spend share * ROAS

# Calculate weekly media contribution
weekly_media = pd.DataFrame()
for ch in channels:
    roas = classical_roas.loc[ch, 'ROAS'] if ch in classical_roas.index else 0
    weekly_media[ch] = data[ch] * roas

# Total media contribution per week
weekly_media['total'] = weekly_media.sum(axis=1)

# Baseline = sales - media contribution
weekly_decomp['baseline'] = data['sales'] - weekly_media['total']

# For each channel, calculate contribution
for ch in channels:
    weekly_decomp[ch] = weekly_media[ch]

# Ensure no negative values (baseline should be positive)
weekly_decomp['baseline'] = weekly_decomp['baseline'].clip(lower=0)

# Create stacked area chart
fig = go.Figure()

# Color palette
colors = {
    'baseline': '#59A14F',  # Green
    'tv': '#4E79A7',        # Blue
    'youtube': '#F28E2B',   # Orange
    'facebook': '#E15759',  # Red
    'instagram': '#76B7B2', # Teal
    'print_media': '#EDC948',  # Yellow
    'radio': '#B07AA1'      # Purple
}

# Add traces in order (baseline first, then channels)
fig.add_trace(go.Scatter(
    x=weekly_decomp['date'], y=weekly_decomp['baseline'],
    mode='lines', name='Baseline',
    stackgroup='one', fillcolor=colors['baseline'], line=dict(width=0)
))

for ch in channels:
    fig.add_trace(go.Scatter(
        x=weekly_decomp['date'], y=weekly_decomp[ch],
        mode='lines', name=channel_display.get(ch, ch),
        stackgroup='one', fillcolor=colors[ch], line=dict(width=0)
    ))

# Add festival annotations (Diwali: Sep-Nov)
diwali_periods = [
    ('2022-09-15', '2022-11-15'),
    ('2023-09-15', '2023-11-15'),
    ('2024-09-15', '2024-11-15'),
]

for start, end in diwali_periods:
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor="yellow", opacity=0.15,
        layer="below", line_width=0,
        annotation_text="Diwali", annotation_position="top left"
    )

fig.update_layout(
    title="Weekly Sales Decomposition Over Time (Classical MMM)",
    xaxis_title="Date",
    yaxis_title="Sales (₹)",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    height=500,
    template="plotly_white"
)

fig.write_image("../outputs/figures/decomposition_stacked_area.png", scale=2)
fig.show()
print("✅ Saved: ../outputs/figures/decomposition_stacked_area.png")

✅ Saved: ../outputs/figures/decomposition_stacked_area.png


## 5. Macro-Adjusted Baseline Decomposition

In [6]:
# =============================================================================
# Macro-Adjusted Baseline Decomposition
# =============================================================================

# Decompose baseline into:
# 1. Base demand (structural)
# 2. CPI / economic effects
# 3. Festival uplift
# 4. Distribution effects (trade_spend)
# 5. Other (residual)

# Using Robyn decomposition as reference (most detailed)
robyn_decomp_df = robyn_roas.reset_index()
robyn_decomp_df = robyn_decomp_df[robyn_decomp_df['rn'].isin(['(Intercept)', 'cpi', 'festival', 'trade_spend', 'gdp_growth', 'season', 'holiday'])]

# Extract contributions
baseline_components = {}
for idx, row in robyn_decomp_df.iterrows():
    rn = row['rn']
    contrib = row['xDecompAgg']
    baseline_components[rn] = contrib

# Print baseline decomposition
print("=" * 60)
print("BASELINE DECOMPOSITION (from Robyn)")
print("=" * 60)
for comp, val in baseline_components.items():
    pct = val / total_sales * 100
    print(f"  {comp:20s}: ₹{val/1e6:8.2f}M ({pct:6.2f}%)")

# Create pie chart
fig = px.pie(
    values=[abs(v) for v in baseline_components.values()],
    names=list(baseline_components.keys()),
    title="Baseline Sub-Component Breakdown",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=450)

fig.write_image("../outputs/figures/baseline_sub_components.png", scale=2)
fig.show()
print("\n✅ Saved: ../outputs/figures/baseline_sub_components.png")

BASELINE DECOMPOSITION (from Robyn)
  (Intercept)         : ₹  324.30M ( 87.23%)
  season              : ₹    0.00M (  0.00%)
  holiday             : ₹   -0.00M ( -0.00%)
  cpi                 : ₹   37.54M ( 10.10%)
  gdp_growth          : ₹   -4.52M ( -1.22%)
  festival            : ₹    1.01M (  0.27%)
  trade_spend         : ₹    1.29M (  0.35%)



✅ Saved: ../outputs/figures/baseline_sub_components.png


## 6. Framework Consensus Analysis

In [7]:
# =============================================================================
# Framework Consensus Analysis
# =============================================================================

# Compare channel contribution % across all 4 frameworks
framework_comparison = pd.DataFrame({
    'Channel': [channel_display.get(ch, ch) for ch in channels],
    'Classical (%)': [classical_decomp.get(ch, 0)/total_sales*100 for ch in channels],
    'Bayesian (%)': [bayesian_decomp.get(ch, 0)/total_sales*100 for ch in channels],
    'Robyn (%)': [robyn_decomp.get(ch, 0)/total_sales*100 for ch in channels],
    'Meridian (%)': [meridian_decomp.get(ch, 0)/total_sales*100 for ch in channels]
})

# Calculate average and std
framework_comparison['Average (%)'] = framework_comparison[['Classical (%)', 'Bayesian (%)', 'Robyn (%)', 'Meridian (%)']].mean(axis=1)
framework_comparison['Std Dev'] = framework_comparison[['Classical (%)', 'Bayesian (%)', 'Robyn (%)', 'Meridian (%)']].std(axis=1)

# Flag disagreement (>20% relative difference)
framework_comparison['Agreement'] = framework_comparison['Std Dev'] / framework_comparison['Average (%)'].replace(0, np.nan) * 100
framework_comparison['Status'] = framework_comparison['Agreement'].apply(
    lambda x: '⚠️ Disagree' if pd.notna(x) and x > 30 else '✅ Agree'
)

print("=" * 90)
print("CHANNEL CONTRIBUTION COMPARISON ACROSS FRAMEWORKS")
print("=" * 90)
print(framework_comparison.round(2).to_string(index=False))

# Create comparison bar chart
fig = framework_comparison.melt(
    id_vars=['Channel', 'Average (%)', 'Std Dev', 'Agreement', 'Status'],
    value_vars=['Classical (%)', 'Bayesian (%)', 'Robyn (%)', 'Meridian (%)'],
    var_name='Framework',
    value_name='Contribution (%)'
)

fig_bar = px.bar(
    fig, x='Channel', y='Contribution (%)', color='Framework',
    barmode='group', title="Channel Contribution % by Framework",
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig_bar.update_layout(height=450, template="plotly_white")
fig_bar.write_image("../outputs/figures/framework_consensus.png", scale=2)
fig_bar.show()
print("\n✅ Saved: ../outputs/figures/framework_consensus.png")

# Pairwise correlation between frameworks
print("\n" + "=" * 60)
print("PAIRWISE CORRELATION BETWEEN FRAMEWORKS")
print("=" * 60)

frameworks_data = framework_comparison[['Classical (%)', 'Bayesian (%)', 'Robyn (%)', 'Meridian (%)']].T
corr_matrix = frameworks_data.corr()
print(corr_matrix.round(3))

CHANNEL CONTRIBUTION COMPARISON ACROSS FRAMEWORKS
  Channel  Classical (%)  Bayesian (%)  Robyn (%)  Meridian (%)  Average (%)  Std Dev  Agreement      Status
       TV            0.0          0.00       0.48          0.00         0.12     0.24     200.00 ⚠️ Disagree
  YouTube            0.0          0.00       0.38          0.00         0.09     0.19     200.00 ⚠️ Disagree
 Facebook            0.0          0.00       0.25          0.00         0.06     0.13     200.00 ⚠️ Disagree
Instagram            0.0          0.00       0.13          0.00         0.03     0.06     200.00 ⚠️ Disagree
    Print            0.0         19.91       0.18          0.37         5.12     9.86     192.81 ⚠️ Disagree
    Radio            0.0          0.00       0.00          0.00         0.00     0.00        NaN     ✅ Agree



✅ Saved: ../outputs/figures/framework_consensus.png

PAIRWISE CORRELATION BETWEEN FRAMEWORKS
       0      1      2      3      4   5
0  1.000  1.000  1.000  1.000 -0.333 NaN
1  1.000  1.000  1.000  1.000 -0.333 NaN
2  1.000  1.000  1.000  1.000 -0.333 NaN
3  1.000  1.000  1.000  1.000 -0.333 NaN
4 -0.333 -0.333 -0.333 -0.333  1.000 NaN
5    NaN    NaN    NaN    NaN    NaN NaN


## 7. Save Decomposition Outputs

In [8]:
# =============================================================================
# Save Decomposition Outputs
# =============================================================================

# 1. Save unified decomposition DataFrame
unified_decomp = pd.DataFrame({
    'Channel': [channel_display.get(ch, ch) for ch in channels],
    'Classical_Contribution': [classical_decomp.get(ch, 0) for ch in channels],
    'Bayesian_Contribution': [bayesian_decomp.get(ch, 0) for ch in channels],
    'Robyn_Contribution': [robyn_decomp.get(ch, 0) for ch in channels],
    'Meridian_Contribution': [meridian_decomp.get(ch, 0) for ch in channels],
    'Classical_Pct': [classical_decomp.get(ch, 0)/total_sales*100 for ch in channels],
    'Bayesian_Pct': [bayesian_decomp.get(ch, 0)/total_sales*100 for ch in channels],
    'Robyn_Pct': [robyn_decomp.get(ch, 0)/total_sales*100 for ch in channels],
    'Meridian_Pct': [meridian_decomp.get(ch, 0)/total_sales*100 for ch in channels],
})

# Add baseline row
baseline_row = pd.DataFrame({
    'Channel': ['Baseline'],
    'Classical_Contribution': [classical_baseline],
    'Bayesian_Contribution': [bayesian_baseline],
    'Robyn_Contribution': [robyn_baseline],
    'Meridian_Contribution': [meridian_baseline],
    'Classical_Pct': [classical_baseline/total_sales*100],
    'Bayesian_Pct': [bayesian_baseline/total_sales*100],
    'Robyn_Pct': [robyn_baseline/total_sales*100],
    'Meridian_Pct': [meridian_baseline/total_sales*100],
})

unified_decomp = pd.concat([baseline_row, unified_decomp], ignore_index=True)

# Save to parquet
unified_decomp.to_parquet('../outputs/models/decomposition.parquet', index=False)
print("✅ Saved: ../outputs/models/decomposition.parquet")

# 2. Save weekly decomposition
weekly_decomp.to_csv('../outputs/models/weekly_decomposition.csv', index=False)
print("✅ Saved: ../outputs/models/weekly_decomposition.csv")

# 3. Save framework comparison
comparison_df.to_csv('../outputs/models/baseline_media_split.csv', index=False)
print("✅ Saved: ../outputs/models/baseline_media_split.csv")

framework_comparison.to_csv('../outputs/models/framework_consensus.csv', index=False)
print("✅ Saved: ../outputs/models/framework_consensus.csv")

# Summary
print("\n" + "=" * 70)
print("📊 SALES DECOMPOSITION SUMMARY")
print("=" * 70)
print(f"Total Sales (156 weeks): ₹{total_sales/1e6:.2f}M")
print(f"\nAverage across 4 frameworks:")
print(f"  • Baseline: {avg_baseline:.1f}%")
print(f"  • Media-Driven: {avg_media:.1f}%")
print(f"\nTop performing channel (avg contribution):")
top_channel = framework_comparison.loc[framework_comparison['Average (%)'].idxmax()]
print(f"  • {top_channel['Channel']}: {top_channel['Average (%)']:.2f}%")
print(f"\nOutputs saved to:")
print(f"  • ../outputs/models/decomposition.parquet")
print(f"  • ../outputs/models/weekly_decomposition.csv")
print(f"  • ../outputs/figures/decomposition_waterfall.png")
print(f"  • ../outputs/figures/decomposition_stacked_area.png")
print(f"  • ../outputs/figures/baseline_sub_components.png")
print(f"  • ../outputs/figures/framework_consensus.png")

✅ Saved: ../outputs/models/decomposition.parquet
✅ Saved: ../outputs/models/weekly_decomposition.csv
✅ Saved: ../outputs/models/baseline_media_split.csv
✅ Saved: ../outputs/models/framework_consensus.csv

📊 SALES DECOMPOSITION SUMMARY
Total Sales (156 weeks): ₹371.77M

Average across 4 frameworks:
  • Baseline: 94.6%
  • Media-Driven: 5.4%

Top performing channel (avg contribution):
  • Print: 5.12%

Outputs saved to:
  • ../outputs/models/decomposition.parquet
  • ../outputs/models/weekly_decomposition.csv
  • ../outputs/figures/decomposition_waterfall.png
  • ../outputs/figures/decomposition_stacked_area.png
  • ../outputs/figures/baseline_sub_components.png
  • ../outputs/figures/framework_consensus.png
